In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import transformers


In [ ]:
pip install rouge_score sacrebleu

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.4 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0ca5e9504c60258cd390d5ae1f4bfb71ae2f0cc5adfc095c37c87cd116f29a66
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [ ]:

import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from rouge_score import rouge_scorer
import sacrebleu
import numpy as np

# Custom Dataset class
class SummarizationDataset(Dataset):
    def __init__(self, texts, summaries, tokenizer, max_input_length=512, max_target_length=128):
        self.texts = texts
        self.summaries = summaries
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        summary = self.summaries[idx]

        # Tokenize input and target
        input_encoding = self.tokenizer(
            text,
            max_length=self.max_input_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        target_encoding = self.tokenizer(
            summary,
            max_length=self.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": input_encoding["input_ids"].squeeze(),
            "attention_mask": input_encoding["attention_mask"].squeeze(),
            "labels": target_encoding["input_ids"].squeeze()
        }

# Load dataset
def load_dataset(file_path):
    df = pd.read_csv(file_path)
    print(f"Total rows in dataset: {len(df)}")

    # Kiểm tra và làm sạch dữ liệu
    invalid_rows = []
    for idx, row in df.iterrows():
        if not isinstance(row["full_text"], str) or not isinstance(row["summary"], str):
            invalid_rows.append((idx, row["full_text"], row["summary"]))
        elif row["full_text"].strip() == "" or row["summary"].strip() == "":
            invalid_rows.append((idx, row["full_text"], row["summary"]))

    if invalid_rows:
        print("Invalid rows found:")
        for idx, text, summary in invalid_rows:
            print(f"Row {idx}: full_text={text} (type: {type(text)}), summary={summary} (type: {type(summary)})")

    # Loại bỏ các hàng không hợp lệ
    df = df.dropna(subset=["full_text", "summary"])
    df = df[df["full_text"].apply(lambda x: isinstance(x, str) and x.strip() != "")]
    df = df[df["summary"].apply(lambda x: isinstance(x, str) and x.strip() != "")]

    print(f"Rows after cleaning: {len(df)}")
    return df["full_text"].tolist(), df["summary"].tolist()

# Compute evaluation metrics (ROUGE and BLEU)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in labels (used for padding in loss computation) with pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = {"rouge1": 0, "rouge2": 0, "rougeL": 0}
    for pred, ref in zip(decoded_preds, decoded_labels):
        scores = scorer.score(ref, pred)
        for key in rouge_scores:
            rouge_scores[key] += scores[key].fmeasure
    for key in rouge_scores:
        rouge_scores[key] /= len(decoded_preds)

    # Compute BLEU score
    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels]).score

    return {
        "rouge1": rouge_scores["rouge1"],
        "rouge2": rouge_scores["rouge2"],
        "rougeL": rouge_scores["rougeL"],
        "bleu": bleu
    }

# Main function to train and save the model
def train_bartpho(file_path, output_dir="/content/drive/MyDrive/Project_NLP/ThanQuocTuan/bartpho_summarization", model_name="vinai/bartpho-syllable"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load tokenizer and model
    try:
        global tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    except Exception as e:
        print(f"Error loading model/tokenizer: {e}")
        print("Ensure you have the correct model name or access to private repo with `huggingface-cli login`")
        raise

    # Load dataset
    try:
        texts, summaries = load_dataset(file_path)
    except Exception as e:
        print(f"Error loading dataset: {e}")
        raise

    # Split dataset into train and validation
    train_texts, val_texts, train_summaries, val_summaries = train_test_split(
        texts, summaries, test_size=0.2, random_state=42
    )

    # Create datasets
    train_dataset = SummarizationDataset(train_texts, train_summaries, tokenizer)
    val_dataset = SummarizationDataset(val_texts, val_summaries, tokenizer)

    # Define training arguments
    try:
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=3,
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            warmup_steps=500,
            weight_decay=0.01,
            logging_dir="./logs",
            logging_steps=10,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            fp16=torch.cuda.is_available(),
            metric_for_best_model="rougeL",
            greater_is_better=True
        )
    except Exception as e:
        print(f"Error initializing TrainingArguments: {e}")
        raise

    # Initialize trainer with compute_metrics
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    # Train the model
    try:
        trainer.train()
    except Exception as e:
        print(f"Error during training: {e}")
        raise

    # Save the model and tokenizer
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model and tokenizer saved to {output_dir}")

if __name__ == "__main__":
    # Example dataset file path (replace with your dataset path)
    dataset_path = "/content/drive/MyDrive/Project_NLP/TranMinhDuy/vnexpress_articles_merge.csv"
    print(transformers.__version__)
    # Train the model
    train_bartpho(dataset_path)


4.53.2
Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tuanb2207574 (tuanb2207574-ctu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss


Error during training: CUDA out of memory. Tried to allocate 3.13 GiB. GPU 0 has a total capacity of 14.74 GiB of which 2.88 GiB is free. Process 7147 has 11.86 GiB memory in use. Of the allocated memory 7.89 GiB is allocated by PyTorch, and 3.83 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


OutOfMemoryError: CUDA out of memory. Tried to allocate 3.13 GiB. GPU 0 has a total capacity of 14.74 GiB of which 2.88 GiB is free. Process 7147 has 11.86 GiB memory in use. Of the allocated memory 7.89 GiB is allocated by PyTorch, and 3.83 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [5]:
def summarize_text(texts, output_dir="/content/drive/MyDrive/Project_NLP/ThanQuocTuan/bartpho_summarization", max_input_length=512, max_target_length=128, batch_size=4):
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    if isinstance(texts, str):
        texts = [texts]
        single_input = True
    else:
        single_input = False
    summaries = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(
                batch_texts,
                max_length=max_input_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            ).to(model.device)
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=max_target_length,
                num_beams=4,
                early_stopping=True
            )
            summaries.extend(tokenizer.batch_decode(summary_ids, skip_special_tokens=True))
            torch.cuda.empty_cache()
    del model
    del tokenizer
    torch.cuda.empty_cache()
    return summaries[0] if single_input else summaries



In [7]:
# Example usage for summarization
test_texts = [
    "Năm 2013, Nguyễn Tính bắt đầu tiếp quản mô hình hộ kinh doanh nuôi thủy sản của gia đình tại Đồng Tháp. Suốt 10 năm gắn bó với nghề, anh chứng kiến không ít bất cập, đặc biệt là tình trạng lạm dụng kháng sinh trong chăn nuôi, ảnh hưởng đến chất lượng thủy sản và môi trường. Trăn trở với điều đó, anh âm thầm tìm kiếm một hướng đi khác biệt, bền vững hơn. Đến năm 2023, sau nhiều năm tích lũy kinh nghiệm và tự nghiên cứu, Tính cùng các cộng sự chính thức thành lập Alpha Amin, với mục tiêu phát triển giải pháp phòng bệnh cho cá rô phi không dùng kháng sinh. Giai đoạn đầu, nhóm sáng lập vừa đảm nhiệm phần kỹ thuật, vừa trực tiếp thử nghiệm sản phẩm tại các hộ nuôi nhỏ lẻ trong tỉnh, từng bước hoàn thiện quy trình trước khi mở rộng quy mô."
]
summaries = summarize_text(test_texts)
for text, summary in zip(test_texts, summaries):
    print(f"Text: {text[:50]}... -> Summary: {summary}")


Text: Năm 2013, Nguyễn Tính bắt đầu tiếp quản mô hình hộ... -> Summary: Đồng ThápNguyễn Tính cùng các cộng sự thành lập Alpha Amin, bắt đầu quản mô hình hộ kinh doanh nuôi thủy sản của gia đình tại Đồng Tháp.
